# Airthereal SD300 — full 1:40 Ryan narration edition

**Private notebook.** This rebuilds only the narration and audio mix. It preserves the approved 100-second 720×1280 video pixels and boot-to-night-mode visual ending by copying the video stream. It transcribes the actual approved movie, synthesizes Ryan with the **Qwen3-TTS 1.7B CustomVoice** checkpoint, then replaces the earlier speech while retaining original operating sounds between speech sections.

1. Set **GPU T4 x2** and **Internet on** in Kaggle Settings.
2. Upload the original `Airthereal_SD300_Final_Approved_Boot_to_Night_Mode_1m40s.mp4` privately using **Input → Upload** (or otherwise place it in Kaggle's input folder). Do not make it public.
3. Run All. The transcript/scene timings print before audio generation.
4. Download the resulting MP4 and ZIP from `/kaggle/working/` and save to [Google Drive Qwen3-TTS — Kaggle Voice Studio](your private Google Drive project folder).

All audio chunks are individually saved so a later failure doesn't erase completed Ryan recordings. The source video is never overwritten.

In [ ]:
%pip -q install -U qwen-tts soundfile faster-whisper librosa

In [ ]:
import os, json, re, gc, math, time, subprocess, shutil, zipfile
from pathlib import Path
import numpy as np, torch, soundfile as sf
from IPython.display import display, Audio

print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')
if not torch.cuda.is_available(): raise RuntimeError('Enable GPU T4 x2 before running.')
OUT=Path('/kaggle/working/ryan_full_video')
OUT.mkdir(parents=True,exist_ok=True)
VIDEO_NAME='Airthereal_SD300_Final_Approved_Boot_to_Night_Mode_1m40s.mp4'
FOUND=[]
for folder in ['/kaggle/input','/kaggle/working','/kaggle/temp']:
    p=Path(folder)
    if p.is_dir():
        FOUND.extend(p.rglob(VIDEO_NAME))
if not FOUND:
    raise FileNotFoundError('Approved source video not attached. Upload MP4 using notebook Input -> Upload first; do not use a different video.')
VIDEO=FOUND[0]
print('SOURCE:',VIDEO, VIDEO.stat().st_size,'bytes')
VIDEO_INFO=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_format','-show_streams','-of','json',str(VIDEO)]))
DURATION=float(VIDEO_INFO['format']['duration'])
print('Video duration:',DURATION,'s; streams:',[(x.get('codec_type'),x.get('codec_name'),x.get('width'),x.get('height')) for x in VIDEO_INFO['streams']])
if not (95 < DURATION < 105): raise ValueError('Unexpected duration. Preserve the 1:40 approved master.')
WAV_IN=OUT/'approved_original_audio_16k.wav'
subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-i',str(VIDEO),'-vn','-ac','1','-ar','16000',str(WAV_IN)],check=True)

In [ ]:
# Transcribe the existing approved narration, with timestamps; no invented product claims.
from faster_whisper import WhisperModel
asr=WhisperModel('medium.en',device='cuda',compute_type='float16')
segments,info=asr.transcribe(str(WAV_IN),beam_size=5,language='en',vad_filter=True,word_timestamps=True,
         initial_prompt='Airthereal SD300 boot and glove dryer review. Boot Mode. Night Mode.',condition_on_previous_text=False)
raw=[{'start':float(s.start),'end':float(s.end),'text':s.text.strip(), 'words':[{'start':float(w.start) if w.start is not None else None,'end':float(w.end) if w.end is not None else None,'word':w.word} for w in (s.words or [])]} for s in segments if s.text.strip()]
(OUT/'approved_original_transcript.json').write_text(json.dumps(raw,indent=2),encoding='utf-8')
print('RECOGNIZED ORIGINAL NARRATION:')
for i,s in enumerate(raw): print(f"{i+1:02d} [{s['start']:05.1f}-{s['end']:05.1f}] {s['text']}")
if not raw: raise RuntimeError('The approved clip had no reliably detected narration; stop rather than invent words.')
del asr
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Organize the existing narration into scene-length chunks. Align against original timed speech.
# Whisper may split sentences differently than the earlier 7 hand-timed narration sections.
# Keep the original transcript words in order and split at punctuation or longer pauses.
import re
allwords=[]
for s in raw:
    if s['words']:
        for w in s['words']:
            if w['start'] is not None and w['end'] is not None and w['end']>w['start']:
                allwords.append({'start':w['start'],'end':w['end'],'word':w['word']})
if len(allwords)<8:
    # fallback to Whisper-level segments
    chunks=[{'start':s['start'],'end':s['end'],'text':s['text']} for s in raw]
else:
    # Build phrase-like chunks with max ~20s and pause-based boundaries.
    breaks=[]
    for i in range(len(allwords)-1):
        gap=allwords[i+1]['start']-allwords[i]['end']
        punct=bool(re.search(r'[.!?]$',allwords[i]['word'].strip()))
        if gap>1.15 or (punct and gap>0.25): breaks.append((i,gap))
    borders=[-1]+[i for i,gap in breaks]+[len(allwords)-1]
    chunks=[]
    for st,ed in zip(borders,borders[1:]):
        ww=allwords[st+1:ed+1]
        if not ww: continue
        chunks.append({'start':ww[0]['start'],'end':ww[-1]['end'],'text':''.join(w['word'] for w in ww).strip()})
    # Merge very short phrases where nearby, keeping a natural seven-section structure if possible.
    while len(chunks)>7:
        costs=[]
        for k in range(len(chunks)-1):
            a,b=chunks[k:k+2]
            gap=b['start']-a['end']
            length=b['end']-a['start']
            cost=(0 if length<=22 else (length-22)*5)+max(0,gap)*3+abs(length-13)*0.03
            costs.append((cost,k))
        _,k=min(costs)
        a,b=chunks[k:k+2]
        chunks[k:k+2]=[{'start':a['start'],'end':b['end'],'text':(a['text']+' '+b['text']).strip()}]
    while len(chunks)<7:
        candidates=[(c['end']-c['start'],i) for i,c in enumerate(chunks) if len(c['text'].split())>=12]
        if not candidates: break
        _,k=max(candidates)
        c=chunks[k]
        ww=[w for w in allwords if w['start']>=c['start']-0.02 and w['end']<=c['end']+0.02]
        if len(ww)<10: break
        mid=len(ww)//2
        left={'start':ww[0]['start'],'end':ww[mid-1]['end'],'text':''.join(w['word'] for w in ww[:mid]).strip()}
        right={'start':ww[mid]['start'],'end':ww[-1]['end'],'text':''.join(w['word'] for w in ww[mid:]).strip()}
        chunks[k:k+1]=[left,right]
# Fix frequent brand spelling without changing product claims.
for c in chunks:
    c['text']=re.sub(r'(?i)\bair\s*(?:thereal|therial|th?ereal|cereal)\b','Airthereal',c['text'])
    c['text']=re.sub(r'(?i)\bSD\s*300\b','SD300',c['text'])
print('REPLACEMENT SECTION PLAN:')
for i,c in enumerate(chunks,1): print(f'{i}: {c["start"]:.2f} - {c["end"]:.2f}: {c["text"]}')
(OUT/'ryan_sections.json').write_text(json.dumps(chunks,indent=2),encoding='utf-8')
if len(chunks)<3: raise RuntimeError('Insufficient timed sections. Stop for manual review.')

In [ ]:
# Load official Ryan 1.7B voice; use the chosen comparison-voice character.
from qwen_tts import Qwen3TTSModel
MODEL_ID='Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice'
STYLE='Warm, natural, conversational male product-review narration. Relaxed, steady pacing; smooth slightly deeper tone; minimal nasal emphasis. Use natural short pauses. No exaggerated announcer voice.'
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model=Qwen3TTSModel.from_pretrained(MODEL_ID,device_map='cuda:0',dtype=DTYPE,attn_implementation='sdpa')
print('Loaded Qwen model:',MODEL_ID, 'voice:', 'Ryan')


In [ ]:
# Generate and preserve each section as a separate 24-kHz WAV before assembling.
from scipy.signal import resample_poly
for i,c in enumerate(chunks,1):
    path=OUT/f'Ryan_section_{i:02}.wav'
    if path.exists() and path.stat().st_size>2048:
        print('Preserving completed:',path.name)
        continue
    print('Rendering:',i,'/',len(chunks),c['text'],flush=True)
    with torch.inference_mode():
        audio,sr=model.generate_custom_voice(text=c['text'],language='English',speaker='Ryan',instruct=STYLE)
    sf.write(path,np.asarray(audio[0],dtype=np.float32),sr)
    print('Saved:',path.name,round(len(audio[0])/sr,2),'sec',flush=True)
# Avoid model taking up GPU memory after rendering.
del model
gc.collect();torch.cuda.empty_cache()

In [ ]:
# Assemble a precisely 100.076-sec narration. Favor natural speed and move pauses around rather than speeding speech.
from scipy.signal import resample_poly
SAMPLE_RATE=48000
full_len=int(round(DURATION*SAMPLE_RATE))
voice=np.zeros(full_len,np.float32)
voice_intervals=[]
prev_end=0.
for i,c in enumerate(chunks,1):
    clip,sr=sf.read(OUT/f'Ryan_section_{i:02}.wav',dtype='float32')
    if clip.ndim==2: clip=clip.mean(axis=1)
    from math import gcd
    d=gcd(sr,SAMPLE_RATE)
    clip=resample_poly(clip,SAMPLE_RATE//d,sr//d).astype(np.float32)
    # First-pass available slot; early starts protect natural speech pacing.
    want=max(0.,c['start']-0.2)
    start=max(want,prev_end+0.12)
    length=len(clip)/SAMPLE_RATE
    if start+length>DURATION-0.4:
        start=max(prev_end+0.08,DURATION-0.4-length)
    if start+length>DURATION-0.05:
        raise RuntimeError(f'Ryan section {i} exceeds the 1:40 runtime; review timing, do not truncate speech.')
    # Gentle 60ms fades prevent clicks.
    ramp=min(int(.06*SAMPLE_RATE),len(clip)//3)
    if ramp:
        clip[:ramp]*=np.linspace(0,1,ramp)
        clip[-ramp:]*=np.linspace(1,0,ramp)
    a=int(start*SAMPLE_RATE); b=a+len(clip)
    voice[a:b]+=clip
    voice_intervals.append({'section':i,'start':start,'end':start+length,'original_start':c['start'],'original_end':c['end'],'text':c['text']})
    prev_end=start+length
print('Ryan placement:',[(x['section'],round(x['start'],1),round(x['end'],1)) for x in voice_intervals])
(OUT/'ryan_timing_manifest.json').write_text(json.dumps(voice_intervals,indent=2),encoding='utf-8')
sf.write(OUT/'SD300_Ryan_1_7B_full_narration.wav',voice,SAMPLE_RATE)
# Quiet original spoken audio during replacement voice, keeping real dryer audio in the gaps.
ORIG48=OUT/'approved_original_audio_48k.wav'
subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-i',str(VIDEO),'-vn','-ac','1','-ar',str(SAMPLE_RATE),str(ORIG48)],check=True)
bg,sr=sf.read(ORIG48,dtype='float32')
if len(bg)<full_len: bg=np.pad(bg,(0,full_len-len(bg)))
bg=bg[:full_len]
gain=np.ones(full_len,np.float32)
for c in raw:
    a=max(0,int((c['start']-0.2)*SAMPLE_RATE))
    b=min(full_len,int((c['end']+0.2)*SAMPLE_RATE))
    gain[a:b]=0.035
# Also attenuate in the exact new Ryan speech windows; feather 250ms edges.
for c in voice_intervals:
    a=max(0,int((c['start']-0.05)*SAMPLE_RATE));b=min(full_len,int((c['end']+0.05)*SAMPLE_RATE));gain[a:b]=0.035
from scipy.ndimage import gaussian_filter1d
gain=gaussian_filter1d(gain,sigma=int(.08*SAMPLE_RATE))
mix=np.clip(bg*gain + voice*.91,-.99,.99)
sf.write(OUT/'SD300_Ryan_1_7B_final_audio_mix.wav',mix,SAMPLE_RATE)


In [ ]:
# Mux video stream without re-encoding; original cut, framing, and footage remain pixel-identical.
FINAL=Path('/kaggle/working/Airthereal_SD300_Qwen3TTS_1_7B_Ryan_Full_1m40s.mp4')
subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-i',str(VIDEO),'-i',str(OUT/'SD300_Ryan_1_7B_final_audio_mix.wav'),'-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','192k','-ar','48000','-t',f'{DURATION:.3f}','-movflags','+faststart',str(FINAL)],check=True)
info=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration,size','-of','json',str(FINAL)]))
print('FINISHED VIDEO:',str(FINAL),info)
if abs(float(info['format']['duration'])-DURATION)>.6: raise RuntimeError('Final video duration mismatch')
with zipfile.ZipFile('/kaggle/working/SD300_Ryan_Full_Audio_and_Transcript.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*'):
        if p.name!='approved_original_audio_48k.wav' and p.name!='approved_original_audio_16k.wav':
            z.write(p,arcname=p.name)
print('Outputs are in /kaggle/working/. Download finished MP4 and upload into the Qwen3-TTS Drive folder. Do not overwrite the original master.')